# Test Kết Nối & Truy Vấn Supabase Database

Notebook này hỗ trợ nhóm dự án:
1. **Kết nối** với database server của Supabase bằng:
   - **Direct PostgreSQL Connection** qua driver `pg8000` (được dùng trong dự án để kết nối qua cổng SSL).
   - **SQLAlchemy Engine** kết hợp với `pandas` để load dữ liệu trực tiếp vào DataFrame.
   - **Supabase API Client (`supabase-py`)** cho các bảng nằm ở schema `public` (tuân thủ Row-Level Security - RLS).
2. **Liệt kê** danh sách toàn bộ các **Schema** và **Tables / Views** trong database để xem cấu trúc thực thể.
3. **Truy vấn mẫu** trên bảng dữ liệu thời tiết thô được chọn: **`staging.stg_open_meteo_weather_raw`**.

In [1]:
import os
import pg8000
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from supabase import create_client, Client

# 1. Load các biến môi trường từ file .env ở thư mục gốc của dự án
# Vì file notebook nằm trong folder 'notebooks', root của dự án nằm ở cấp cha (parents[0])
ROOT_DIR = Path().resolve().parents[0]
env_path = ROOT_DIR / ".env"
load_dotenv(env_path)

print(f"[INFO] Đã tải file cấu hình môi trường tại: {env_path}")

[INFO] Đã tải file cấu hình môi trường tại: D:\Learning\FPT_polytechnic\Sem6\datn_outlier_hs_nlmt\.env


## 1. Kết nối Direct PostgreSQL (pg8000)
Chúng ta kết nối trực tiếp vào PostgreSQL server thông qua cổng giao tiếp an toàn (SSL) và driver `pg8000` như đã định nghĩa trong dự án.

In [4]:
# Lấy thông tin cấu hình từ file .env
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

if not all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    raise ValueError("Thiếu thông tin cấu hình cơ sở dữ liệu trong file .env")
print("[INFO] Đã lấy thông tin cấu hình cơ sở dữ liệu từ file .env")

[INFO] Đã lấy thông tin cấu hình cơ sở dữ liệu từ file .env


In [5]:
# Tạo kết nối PostgreSQL
conn = pg8000.connect(
    host=DB_HOST,
    port=int(DB_PORT) if DB_PORT else 5432,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    ssl_context=True,
)
cur = conn.cursor()
print("[OK] Kết nối trực tiếp đến database thành công!")

[OK] Kết nối trực tiếp đến database thành công!


### 1.1 Liệt kê các Schema trong Database
Truy vấn schema hệ thống loại bỏ các schema mặc định của postgres để lấy ra các schema phục vụ cho dự án.

In [6]:
cur.execute("""
    SELECT schema_name 
    FROM information_schema.schemata
    WHERE schema_name NOT LIKE 'pg_%' AND schema_name != 'information_schema'
    ORDER BY schema_name;
""")
schemas = cur.fetchall()

print("=== CÁC SCHEMAS HIỆN CÓ ===")
for s in schemas:
    print(f"- {s[0]}")

=== CÁC SCHEMAS HIỆN CÓ ===
- auth
- extensions
- graphql
- graphql_public
- public
- realtime
- staging
- storage
- vault


### 1.2 Liệt kê các Tables & Views hiện có
Hiển thị chi tiết từng bảng/view được phân chia theo schema.

In [7]:
cur.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema NOT LIKE 'pg_%' AND table_schema != 'information_schema'
    ORDER BY table_schema, table_name;
""")
tables = cur.fetchall()

print("=== DANH SÁCH BẢNG & VIEWS ===")
current_schema = ""
for t_schema, t_name, t_type in tables:
    if t_schema != current_schema:
        current_schema = t_schema
        print(f"\n[Schema: {current_schema.upper()}]")
    print(f"  ├─ {t_name} ({t_type})")

=== DANH SÁCH BẢNG & VIEWS ===

[Schema: AUTH]
  ├─ audit_log_entries (BASE TABLE)
  ├─ custom_oauth_providers (BASE TABLE)
  ├─ flow_state (BASE TABLE)
  ├─ identities (BASE TABLE)
  ├─ instances (BASE TABLE)
  ├─ mfa_amr_claims (BASE TABLE)
  ├─ mfa_challenges (BASE TABLE)
  ├─ mfa_factors (BASE TABLE)
  ├─ oauth_authorizations (BASE TABLE)
  ├─ oauth_client_states (BASE TABLE)
  ├─ oauth_clients (BASE TABLE)
  ├─ oauth_consents (BASE TABLE)
  ├─ one_time_tokens (BASE TABLE)
  ├─ refresh_tokens (BASE TABLE)
  ├─ saml_providers (BASE TABLE)
  ├─ saml_relay_states (BASE TABLE)
  ├─ schema_migrations (BASE TABLE)
  ├─ sessions (BASE TABLE)
  ├─ sso_domains (BASE TABLE)
  ├─ sso_providers (BASE TABLE)
  ├─ users (BASE TABLE)
  ├─ webauthn_challenges (BASE TABLE)
  ├─ webauthn_credentials (BASE TABLE)

[Schema: EXTENSIONS]
  ├─ pg_stat_statements (VIEW)
  ├─ pg_stat_statements_info (VIEW)

[Schema: PUBLIC]
  ├─ dim_date (BASE TABLE)
  ├─ dim_geography (BASE TABLE)
  ├─ dim_solar_site (BAS

## 2. Kết nối bằng SQLAlchemy Engine & Pandas
Để thuận tiện cho phân tích và trực quan hóa dữ liệu, SQLAlchemy được dùng để tạo một connection engine có hỗ trợ SSL, giúp Pandas nạp dữ liệu SQL trực tiếp vào DataFrame.

In [8]:
# Tạo Connection URI sử dụng driver pg8000
db_uri = f"postgresql+pg8000://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Khởi tạo SQLAlchemy Engine với SSL cấu hình qua connect_args
engine = create_engine(db_uri, connect_args={"ssl_context": True})
print("[OK] Đã tạo SQLAlchemy Engine!")

[OK] Đã tạo SQLAlchemy Engine!


## 3. Truy vấn bảng chọn lọc: `staging.stg_open_meteo_weather_raw`

Bảng này chứa dữ liệu dự báo/thực tế thời tiết thô lấy từ Open-Meteo API. Chúng ta sẽ thực hiện:
1. Đọc preview 10 dòng đầu.
2. Đọc tập dữ liệu mẫu lớn hơn (2000 dòng).
3. Ép kiểu và thực hiện thống kê mô tả (Descriptive Statistics) các trường thông tin thời tiết chính (Nhiệt độ, bức xạ mặt trời, lượng mưa, tốc độ gió, độ bao phủ mây).

In [9]:
# 3.1 Xem trước 10 dòng dữ liệu thời tiết
query_limit = """
    SELECT * 
    FROM staging.stg_open_meteo_weather_raw 
    LIMIT 10;
"""
# Đọc bằng pandas
df_preview = pd.read_sql_query(query_limit, con=engine)
print(f"Kích thước dữ liệu mẫu: {df_preview.shape}")
df_preview.head()

Kích thước dữ liệu mẫu: (10, 17)


,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,temperature_2m,weather_code,is_day,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high,wind_speed_10m,precipitation,sunshine_duration,sitekey,latitude,longitude
0,2020-01-01 00:00:00,0.0,0.0,0.0,17.6,0,0,0,0,0,0,2.3,0.0,0.0,1,-36.11120917,146.8486788
1,2020-01-01 01:00:00,0.0,0.0,0.0,15.6,0,0,0,0,0,0,3.4,0.0,0.0,1,-36.11120917,146.8486788
2,2020-01-01 02:00:00,0.0,0.0,0.0,14.8,0,1,0,0,0,0,3.0,0.0,0.0,1,-36.11120917,146.8486788
3,2020-01-01 03:00:00,50.0,25.0,25.0,16.4,0,1,0,0,0,0,0.8,0.0,3409.89,1,-36.11120917,146.8486788
4,2020-01-01 04:00:00,225.0,156.0,69.0,18.2,0,1,0,0,0,0,1.6,0.0,3600.0,1,-36.11120917,146.8486788


In [10]:
# 3.2 Tải 2000 dòng để phân tích và chuẩn hóa kiểu dữ liệu
query_analysis = """
    SELECT sitekey, timestamp, temperature_2m, shortwave_radiation, wind_speed_10m, precipitation, cloud_cover
    FROM staging.stg_open_meteo_weather_raw 
    LIMIT 2000;
"""
df_weather = pd.read_sql_query(query_analysis, con=engine)

# Ép kiểu các cột dạng string sang numeric
numeric_cols = ["temperature_2m", "shortwave_radiation", "wind_speed_10m", "precipitation", "cloud_cover"]
for col in numeric_cols:
    df_weather[col] = pd.to_numeric(df_weather[col], errors="coerce")

print("=== THÔNG TIN DỮ LIỆU ĐÃ CHUẨN HÓA KIỂU SỐ ===")
df_weather.info()

=== THÔNG TIN DỮ LIỆU ĐÃ CHUẨN HÓA KIỂU SỐ ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sitekey              2000 non-null   object 
 1   timestamp            2000 non-null   object 
 2   temperature_2m       2000 non-null   float64
 3   shortwave_radiation  2000 non-null   float64
 4   wind_speed_10m       2000 non-null   float64
 5   precipitation        2000 non-null   float64
 6   cloud_cover          2000 non-null   int64  
dtypes: float64(4), int64(1), object(2)
memory usage: 109.5+ KB


In [11]:
# 3.3 Thống kê mô tả dữ liệu thời tiết mẫu
print("=== THỐNG KÊ MÔ TẢ CÁC BIẾN THỜI TIẾT ===")
df_weather.describe()

=== THỐNG KÊ MÔ TẢ CÁC BIẾN THỜI TIẾT ===


,temperature_2m,shortwave_radiation,wind_speed_10m,precipitation,cloud_cover
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,22.011000,269.654500,9.994150,0.095250,38.864500
std,5.972709,334.149772,5.306354,0.508401,38.558923
min,8.900000,0.000000,0.400000,0.000000,0.000000
25%,17.800000,0.000000,6.100000,0.000000,0.750000
50%,21.400000,61.000000,8.700000,0.000000,25.000000
75%,25.900000,546.750000,13.225000,0.000000,80.250000
max,43.200000,1067.000000,31.300000,7.400000,100.000000


In [12]:
# 3.4 Đếm phân phối dữ liệu theo trạm (sitekey)
print("=== SỐ DÒNG DỮ LIỆU THỜI TIẾT THEO TỪNG TRẠM (SITEKEY) ===")
df_weather["sitekey"].value_counts()

=== SỐ DÒNG DỮ LIỆU THỜI TIẾT THEO TỪNG TRẠM (SITEKEY) ===


sitekey
1    2000
Name: count, dtype: int64

## 4. Kết nối qua API Client (Tùy chọn cho schema public)
Supabase cung cấp một API Client rất mạnh hỗ trợ các tác vụ thao tác CRUD thông thường trên client-side cho schema `public`.

In [13]:
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_ANON_KEY = os.getenv("SUPABASE_ANON_KEY")

if SUPABASE_URL and SUPABASE_ANON_KEY:
    try:
        supabase: Client = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)
        print("[OK] Đã kết nối Supabase API Client!")
        
        # Truy vấn demo 3 dòng từ bảng public.test_table hoặc public.vn_solar_site_details
        response = supabase.table("test_table").select("*").limit(3).execute()
        print("Dữ liệu mẫu từ bảng 'test_table':", response.data)
    except Exception as e:
        print(f"[Lưu ý] Không truy vấn được qua API (có thể do cấu hình RLS hoặc bảng trống): {e}")
else:
    print("[WARN] Thiếu thông tin SUPABASE_URL hoặc SUPABASE_ANON_KEY trong file .env")

[OK] Đã kết nối Supabase API Client!
[Lưu ý] Không truy vấn được qua API (có thể do cấu hình RLS hoặc bảng trống): {'message': 'JSON could not be generated', 'code': 401, 'hint': 'Refer to full message for details', 'details': 'b\'{"message":"Invalid API key","hint":"Double check your Supabase `anon` or `service_role` API key."}\''}


## 5. Dọn dẹp kết nối

In [15]:
# Đóng kết nối PostgreSQL thô để giải phóng connection pool
try:
    cur.close()
    conn.close()
    print("[OK] Đã đóng kết nối pg8000 thành công.")
except NameError:
    pass
except Exception as e:
    print(f"Lỗi khi đóng kết nối: {e}")

Lỗi khi đóng kết nối: connection is closed
